# Executive Summary

**Objective:** 
To integrate geographical metadata, resolve string formatting inconsistencies, and engineer normalized metrics, transforming the raw data into a finalized state ready for exploratory data analysis.

**Data Flow:**
*   **Inputs:** 
    * `data\raw\internship_positions.parquet`
    * `data\external\administrative_divisions.parquet`
*   **Output:** `data\interim\internship_postings.parquet` (Exported to the interim directory to preserve datatypes)

**Key Operations Performed:**
1. **Data Integration (Province Mapping) [<u>[click]</u>](#1-data-integration):** Mapped raw job locations to their respective provinces by joining against the administrative divisions dictionary.
    * **String Cleaning [<u>[click]</u>](#11-string-cleaning):** Resolved mismatching geographic keys by standardizing text (lowercasing, stripping punctuation, and removing extraneous words).
    * **Manual Overrides [<u>[click]</u>](#12-manual-overrides):** Applied a manual dictionary mapping to achieve 100% province mapping with zero null values.
2. **Data Conversion [<u>[click]</u>](#2-data-conversion):** Cast `weekly_working_day` to a categorical datatype due to its low cardinality (2 unique values).
3. **Feature Engineering [<u>[click]</u>](#3-feature-engineering):**
    * Engineered the `acceptance_percentage` column (`100 * approved_quota / (1 + applicant_count)`).
    * Binned all heavy right-skewed numericals (`requested_quota`, `approved_quota`, `applicant_count`, `acceptance_percentage`) to capture "whale" postings in an "Extreme" category without deleting them.
    * One-hot encoded `education_level` for downstream stakeholder consumption.
    * Extracted a new binary flag, `allows_all_majors`, by parsing the `job_description` column.
4. **Schema Finalization [<u>[click]</u>](#4-schema-finalization):** Reorganized the final 14 columns into a logical analytical structure before exporting.

# Setup & Imports

In [25]:
# Import libraries
import numpy as np
import pandas as pd

from src.config import EXTERNAL_DATA_DIR, INTERIM_DATA_DIR, RAW_DATA_DIR

In [26]:
# Load datasets
adm_divisions = pd.read_parquet(EXTERNAL_DATA_DIR / "administrative_divisions.parquet")
internship_positions = pd.read_parquet(RAW_DATA_DIR / "internship_positions.parquet")

# 1. Data Integration
Mapping raw job locations to their respective provinces by joining against the administrative divisions dictionary.

In [27]:
# Convert a regency-province table into a dictionary
adm_divisions_dict = dict(zip(adm_divisions["regency"], adm_divisions["province"]))


In [28]:
# Create a new column, `province`, by mapping with a dictionary
internship_positions["province"] = internship_positions["job_location"].map(adm_divisions_dict)
display(internship_positions.head())

,job_id,published_at,job_title,company,job_location,education_level,allowed_major,job_description,weekly_working_day,requested_quota,approved_quota,applicant_count,province
0,a240f2ba-12c0-4958-b416-c3e9c1d4e344,2026-07-16T12:58:55+07:00,PSIKOLOG,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Psikologi,1. Melakukan asesmen psikologis terhadap anak ...,6,1,1,0,Sumatera Utara
1,a240f336-b6b7-47fe-8096-5b4c2eb2ed1f,2026-07-16T12:58:55+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,1,0,Sumatera Utara
2,a242e6a4-2c7a-4ce4-8a57-7aff601a5e2c,2026-07-16T12:57:40+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SALATIGA,Kota Salatiga,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,5,1,1,0,Jawa Tengah
3,a24138aa-bcf6-4940-bfaf-dfe5dbf66ca6,2026-07-16T12:50:42+07:00,PERAWAT KESEHATAN,LEMBAGA PEMASYARAKATAN KELAS III SUKAMARA,Kab. Sukamara,Bachelor,Ilmu Gizi,1. Memberikan perawatan kesehatan umum dan tin...,6,1,1,0,Kalimantan Tengah
4,a23f7c52-9123-46e9-bb5e-be376b1d77f2,2026-07-16T12:38:52+07:00,Psikiater,LEMBAGA PEMASYARAKATAN KELAS III ARJASA,Kab. Sumenep,Bachelor,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,1,0,Jawa Timur


In [29]:
# Find regencies from `internship_positions` that have no match 
# with their corresponding provinces
display(
    internship_positions[
        internship_positions["province"].isnull()
    ].groupby("job_location")["job_id"].count().sort_values(ascending=False)
)

job_location
Kota Batam                          221
Kota Palangkaraya                   120
Kota Dumai                           77
Kab. Pangkajene Kepulauan            57
Kota Bau Bau                         56
Kota Banjarbaru                      48
Kab. Siak                            46
Kota Lubuk Linggau                   43
Kab. Gunungkidul                     42
Kab. Karangasem                      39
Kota Sawahlunto                      38
Kab. Batanghari                      31
Kab. Tulang Bawang                   28
Kota Pematangsiantar                 25
Unknown Location                     24
Kab. Kotabaru                        24
Kab. Banyuasin                       23
Kab. Labuhanbatu                     23
Kota Pare Pare                       20
Kab. Tojo Una Una                    18
Kab. Pahuwato                        17
Kab. Kep. Siau Tagulandang Biaro     17
Kepulauan Tanimbar                   15
Kab Timor Tengah Selatan             15
Kab. Toli Toli             

In [30]:
# Get all regencies from `adm_divisions` that have no match
# with job locations from `internship_positions`
matched_regencies = list(set(internship_positions[
    ~internship_positions.province.isnull()
]["job_location"].to_list()))

unmatched_regencies = list(set(adm_divisions.regency.to_list()) - set(matched_regencies))

display(
    adm_divisions[
        adm_divisions.regency.isin(unmatched_regencies)
    ][["regency", "province"]].sort_values("regency", ascending=False)
)

,regency,province
70,Kota Sawah Lunto,Sumatera Barat
50,Kota Pematang Siantar,Sumatera Utara
420,Kota Parepare,Sulawesi Selatan
341,Kota Palangka Raya,Kalimantan Tengah
114,Kota Lubuklinggau,Sumatera Selatan
...,...,...
90,Kab. Batang Hari,Jambi
104,Kab. Banyu Asin,Sumatera Selatan
144,Kab. Bangka Selatan,Kepulauan Bangka Belitung
385,Kab. Banggai Kepulauan,Sulawesi Tengah


## 1.1 String Cleaning

In [31]:
# Fallback mapping: Stripping punctuation and whitespace resolves mismatches
# caused by inconsistent data entry on the scraped platform.
plain_adm_div_dict = dict(
    zip(
        adm_divisions["regency"]
        .str.lower()
        .str.replace(r"\sdan\s", " ", case=False, regex=True)
        .str.replace(r"\W", "", regex=True),
        adm_divisions["province"],
    )
)

display(plain_adm_div_dict)

mask = internship_positions["province"].isnull()
internship_positions.loc[mask, "province"] = (
    internship_positions.loc[mask, "job_location"]
    .str.lower()
    .str.replace(r"\sdan\s", " ", case=False, regex=True)
    .str.replace(r"\W", "", regex=True)
    .map(plain_adm_div_dict)
)

{'kabsimeulue': 'Aceh',
 'kabacehsingkil': 'Aceh',
 'kabacehselatan': 'Aceh',
 'kabacehtenggara': 'Aceh',
 'kabacehtimur': 'Aceh',
 'kabacehtengah': 'Aceh',
 'kabacehbarat': 'Aceh',
 'kabacehbesar': 'Aceh',
 'kabpidie': 'Aceh',
 'kabbireuen': 'Aceh',
 'kabacehutara': 'Aceh',
 'kabacehbaratdaya': 'Aceh',
 'kabgayolues': 'Aceh',
 'kabacehtamiang': 'Aceh',
 'kabnaganraya': 'Aceh',
 'kabacehjaya': 'Aceh',
 'kabbenermeriah': 'Aceh',
 'kabpidiejaya': 'Aceh',
 'kotabandaaceh': 'Aceh',
 'kotasabang': 'Aceh',
 'kotalangsa': 'Aceh',
 'kotalhokseumawe': 'Aceh',
 'kotasubulussalam': 'Aceh',
 'kabnias': 'Sumatera Utara',
 'kabmandailingnatal': 'Sumatera Utara',
 'kabtapanuliselatan': 'Sumatera Utara',
 'kabtapanulitengah': 'Sumatera Utara',
 'kabtapanuliutara': 'Sumatera Utara',
 'kabtobasamosir': 'Sumatera Utara',
 'kablabuhanbatu': 'Sumatera Utara',
 'kabasahan': 'Sumatera Utara',
 'kabsimalungun': 'Sumatera Utara',
 'kabdairi': 'Sumatera Utara',
 'kabkaro': 'Sumatera Utara',
 'kabdeliserdang': '

In [32]:
# Check for the missing values again by finding regencies from `internship_positions`
# that have no match with their corresponding provinces
display(
    internship_positions[
        internship_positions["province"].isnull()
    ].groupby("job_location")["job_id"].count().sort_values(ascending=False)
)

job_location
Unknown Location                    24
Kab. Kep. Siau Tagulandang Biaro    17
Kab. Pahuwato                       17
Kepulauan Tanimbar                  15
Kab. Mahakam Ulu                     1
Name: job_id, dtype: int64

## 1.2 Manual Overrides

In [33]:
# Manual overrides for edge cases missing from the standard division dataset
manual_dict = {
    "Kab. Kep. Siau Tagulandang Biaro": "Sulawesi Utara",
    "Kab. Mahakam Ulu": "Kalimantan Timur",
    "Kab. Pahuwato": "Gorontalo",
    "Kepulauan Tanimbar": "Maluku",
    "Unknown Location": "Unknown Location",
}

mask = internship_positions["province"].isnull()
internship_positions.loc[mask, "province"] = internship_positions.loc[
    mask, "job_location"
].map(manual_dict)

In [34]:
# Check for the missing values again by finding regencies from `internship_positions`
# that have no match with their corresponding provinces
display(
    internship_positions[
        internship_positions["province"].isnull()
    ].groupby("job_location")["job_id"].count().sort_values(ascending=False)
)

Series([], Name: job_id, dtype: int64)

In [35]:
# Rename Column `job_location` to `regency_city`
internship_positions.rename(columns={"job_location": "regency_city"}, inplace=True)

In [36]:
# Fix some regency and city names
internship_positions["regency_city"] = (
    internship_positions.regency_city
    .str.replace(r"^Kab\s", r"Kab. ", regex=True)
    .str.replace("Pahuwato", "Pohuwato")
    .str.replace(r"^Kepulauan\s", r"Kab. Kep. ", regex=True)
    .str.replace(r"\sKepulauan\s", r" Kep. ", regex=True)
)

# 2. Data Conversion
Casting `weekly_working_day` to a categorical datatype due to its low cardinality (2 unique values).

In [37]:
# Cast the data type of `weekly_working_day` to string
internship_positions = internship_positions.astype({"weekly_working_day": "str"})

internship_positions.info()

<class 'pandas.DataFrame'>
RangeIndex: 28322 entries, 0 to 28321
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   job_id              28322 non-null  str  
 1   published_at        28322 non-null  str  
 2   job_title           28322 non-null  str  
 3   company             28322 non-null  str  
 4   regency_city        28322 non-null  str  
 5   education_level     28322 non-null  str  
 6   allowed_major       28322 non-null  str  
 7   job_description     28322 non-null  str  
 8   weekly_working_day  28322 non-null  str  
 9   requested_quota     28322 non-null  int64
 10  approved_quota      28322 non-null  int64
 11  applicant_count     28322 non-null  int64
 12  province            28322 non-null  str  
dtypes: int64(3), str(10)
memory usage: 20.0 MB


# 3. Feature Engineering

## 3.1 Feature Construction
Constructing the `acceptance_percentage` column (`100 * approved_quota / (1 + applicant_count)`)

In [38]:
# Create Column `acceptance_percentage`
internship_positions["acceptance_percentage"] = round(
    100
    * internship_positions["approved_quota"]
    / (internship_positions["applicant_count"].add(1)),
    2,
)

display(internship_positions.sample(10))

,job_id,published_at,job_title,company,regency_city,education_level,allowed_major,job_description,weekly_working_day,requested_quota,approved_quota,applicant_count,province,acceptance_percentage
20271,a2415f8f-12f1-464a-a73e-dad7bad16ff7,2026-07-16T12:55:33+07:00,Perawat,LEMBAGA PEMASYARAKATAN KELAS I MEDAN,Kota Medan,Bachelor,Keperawatan,1. Memberikan perawatan medis dasar bagi pegaw...,6,1,1,11,Sumatera Utara,8.33
8282,a2412ad5-2088-4211-a0f5-fb9704461879,2026-07-16T12:14:16+07:00,PENGELOLA KEGIATAN KERJA,LEMBAGA PEMASYARAKATAN KELAS IIA LAHAT,Kab. Lahat,Bachelor,"Ilmu Pertanian, Ilmu Perikanan, Budi Daya Tana...",$25,6,7,7,35,Sumatera Selatan,19.44
15883,a241ef6d-372a-4738-ae1f-fe23e276ed9d,2026-07-16T10:51:49+07:00,Sales and Marketing,PT Infiniti Bioanalitika Solusindo,Kota Adm. Jakarta Barat,Bachelor,"Farmasi, Ilmu Pertanian, Peternakan, Bioteknol...",Membantu tim sales untuk handle customer sesua...,5,2,1,8,DKI Jakarta,11.11
15974,a2293a6c-efeb-4155-a400-ec650f47ab35,2026-07-16T10:35:21+07:00,Customer Care Officer,PT. Anugerah Sejahtera Putera,Kab. Sleman,"Diploma, Bachelor","Manajemen, Administrasi, Hubungan Masyarakat, ...",Menjawab pertanyaan customer baik keluhan maup...,6,1,1,8,DI Yogyakarta,11.11
25585,a24361bc-5ecc-4243-9a67-eb7e220b3157,2026-07-16T13:12:37+07:00,Administrasi Klaitan,BPJS Kesehatan Kantor Cabang Jayapura,Kota Jayapura,"Diploma, Bachelor","Keperawatan, Administrasi Rumah Sakit, Farmasi...",Membantu pelaksanaan kegiatan administratif da...,5,1,1,18,Papua,5.26
27229,a243a8f7-d2c3-4431-b8c3-d4ba362f0e40,2026-07-16T12:09:25+07:00,PEMBINAAN KEPRIBADIAN,RUMAH TAHANAN NEGARA KELAS IIB KUDUS,Kab. Kudus,Bachelor,Pendidikan Agama Islam,1.\tMenyusun dan melaksanakan program pembinaa...,6,1,1,24,Jawa Tengah,4.00
5414,a23ff738-3a17-4f12-8113-900d56590887,2026-07-16T09:51:00+07:00,Logistic Assistance Intern,PT Sinarniaga Sejahtera,Kota Adm. Jakarta Timur,"Bachelor, Diploma","Manajemen, sistem informasi, Statistika, PJJ T...",Mendukung pelaksanaan stock opname produk seca...,6,2,2,9,DKI Jakarta,20.00
10490,a224c21b-83ea-4121-b122-70960ddc8baf,2026-07-16T10:43:56+07:00,Magang Admin PPIC,PT. Gunze Indonesia,Kab. Bekasi,Bachelor,"Kimia Tekstil, Teknik Tekstil, Teknologi Kimia...",Membantu pelaksanaan administrasi Production P...,5,1,1,6,Jawa Barat,14.29
28187,a2413ba4-2173-4ea5-b3fa-acab4f056d2f,2026-07-16T12:06:57+07:00,Relationship Officer (RO),BPJS Kesehatan Kantor Cabang Semarang,Kota Semarang,"Diploma, Bachelor","Administrasi BIsnis, Manajemen, Ilmu Komunikas...",Membantu pelaksanaan kegiatan administratif da...,5,1,1,42,Jawa Tengah,2.33
3196,a232ec85-acca-43af-bdff-27d5cdc373a3,2026-07-16T10:33:33+07:00,Sales Intern,PT. Mitra Global Holiday,Kota Adm. Jakarta Pusat,"Diploma, Bachelor","Administrasi BIsnis, Manajemen, Manajemen Bisn...",1. Assist in preparing sales reports and handl...,5,4,4,14,DKI Jakarta,26.67


## 3.2 Feature Transformation
Binning all heavy right-skewed numericals (`requested_quota`, `approved_quota`, `applicant_count`, `acceptance_percentage`) to capture "whale" postings in an "Extreme" category without deleting them.

In [39]:
# Bin `requested_quota` and `approved_quota`
quota_edges = [1, 2, 10, 50, np.inf]
quota_labels = ["1 to 2", "3 to 10", "11 to 50", "50+"]

internship_positions["requested_quota_category"] = pd.cut(
    internship_positions["requested_quota"],
    bins=quota_edges,
    labels=quota_labels,
    include_lowest=True
)

internship_positions["approved_quota_category"] = pd.cut(
    internship_positions["approved_quota"],
    bins=quota_edges,
    labels=quota_labels,
    include_lowest=True
)

display(internship_positions.head())

,job_id,published_at,job_title,company,regency_city,education_level,allowed_major,job_description,weekly_working_day,requested_quota,approved_quota,applicant_count,province,acceptance_percentage,requested_quota_category,approved_quota_category
0,a240f2ba-12c0-4958-b416-c3e9c1d4e344,2026-07-16T12:58:55+07:00,PSIKOLOG,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Psikologi,1. Melakukan asesmen psikologis terhadap anak ...,6,1,1,0,Sumatera Utara,100.0,1 to 2,1 to 2
1,a240f336-b6b7-47fe-8096-5b4c2eb2ed1f,2026-07-16T12:58:55+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,1,0,Sumatera Utara,100.0,1 to 2,1 to 2
2,a242e6a4-2c7a-4ce4-8a57-7aff601a5e2c,2026-07-16T12:57:40+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SALATIGA,Kota Salatiga,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,5,1,1,0,Jawa Tengah,100.0,1 to 2,1 to 2
3,a24138aa-bcf6-4940-bfaf-dfe5dbf66ca6,2026-07-16T12:50:42+07:00,PERAWAT KESEHATAN,LEMBAGA PEMASYARAKATAN KELAS III SUKAMARA,Kab. Sukamara,Bachelor,Ilmu Gizi,1. Memberikan perawatan kesehatan umum dan tin...,6,1,1,0,Kalimantan Tengah,100.0,1 to 2,1 to 2
4,a23f7c52-9123-46e9-bb5e-be376b1d77f2,2026-07-16T12:38:52+07:00,Psikiater,LEMBAGA PEMASYARAKATAN KELAS III ARJASA,Kab. Sumenep,Bachelor,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,1,0,Jawa Timur,100.0,1 to 2,1 to 2


In [40]:
# Bin Column `applicant_count`
applicant_edges = [0, 5, 10, 20, 50, np.inf]
applicant_labels = ["0 to 5", "6 to 10", "11 to 20", "21 to 50", "50+"]

internship_positions["applicant_count_category"] = pd.cut(
    internship_positions["applicant_count"],
    bins=applicant_edges,
    labels=applicant_labels,
    include_lowest=True
)

display(internship_positions.head())

,job_id,published_at,job_title,company,regency_city,education_level,allowed_major,job_description,weekly_working_day,requested_quota,approved_quota,applicant_count,province,acceptance_percentage,requested_quota_category,approved_quota_category,applicant_count_category
0,a240f2ba-12c0-4958-b416-c3e9c1d4e344,2026-07-16T12:58:55+07:00,PSIKOLOG,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Psikologi,1. Melakukan asesmen psikologis terhadap anak ...,6,1,1,0,Sumatera Utara,100.0,1 to 2,1 to 2,0 to 5
1,a240f336-b6b7-47fe-8096-5b4c2eb2ed1f,2026-07-16T12:58:55+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,1,0,Sumatera Utara,100.0,1 to 2,1 to 2,0 to 5
2,a242e6a4-2c7a-4ce4-8a57-7aff601a5e2c,2026-07-16T12:57:40+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SALATIGA,Kota Salatiga,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,5,1,1,0,Jawa Tengah,100.0,1 to 2,1 to 2,0 to 5
3,a24138aa-bcf6-4940-bfaf-dfe5dbf66ca6,2026-07-16T12:50:42+07:00,PERAWAT KESEHATAN,LEMBAGA PEMASYARAKATAN KELAS III SUKAMARA,Kab. Sukamara,Bachelor,Ilmu Gizi,1. Memberikan perawatan kesehatan umum dan tin...,6,1,1,0,Kalimantan Tengah,100.0,1 to 2,1 to 2,0 to 5
4,a23f7c52-9123-46e9-bb5e-be376b1d77f2,2026-07-16T12:38:52+07:00,Psikiater,LEMBAGA PEMASYARAKATAN KELAS III ARJASA,Kab. Sumenep,Bachelor,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,1,0,Jawa Timur,100.0,1 to 2,1 to 2,0 to 5


In [41]:
# Bin Column `acceptance_percentage`
acceptance_edges = [0, 10, 25, 50, np.inf]
acceptance_labels = ["0 - 10%", "11 - 25%", "26 - 50%", "50%+"]

internship_positions["acceptance_percentage_category"] = pd.cut(
    internship_positions["acceptance_percentage"],
    bins=acceptance_edges,
    labels=acceptance_labels,
    include_lowest=True
)

display(internship_positions.head())

,job_id,published_at,job_title,company,regency_city,education_level,allowed_major,job_description,weekly_working_day,requested_quota,approved_quota,applicant_count,province,acceptance_percentage,requested_quota_category,approved_quota_category,applicant_count_category,acceptance_percentage_category
0,a240f2ba-12c0-4958-b416-c3e9c1d4e344,2026-07-16T12:58:55+07:00,PSIKOLOG,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Psikologi,1. Melakukan asesmen psikologis terhadap anak ...,6,1,1,0,Sumatera Utara,100.0,1 to 2,1 to 2,0 to 5,50%+
1,a240f336-b6b7-47fe-8096-5b4c2eb2ed1f,2026-07-16T12:58:55+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,1,0,Sumatera Utara,100.0,1 to 2,1 to 2,0 to 5,50%+
2,a242e6a4-2c7a-4ce4-8a57-7aff601a5e2c,2026-07-16T12:57:40+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SALATIGA,Kota Salatiga,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,5,1,1,0,Jawa Tengah,100.0,1 to 2,1 to 2,0 to 5,50%+
3,a24138aa-bcf6-4940-bfaf-dfe5dbf66ca6,2026-07-16T12:50:42+07:00,PERAWAT KESEHATAN,LEMBAGA PEMASYARAKATAN KELAS III SUKAMARA,Kab. Sukamara,Bachelor,Ilmu Gizi,1. Memberikan perawatan kesehatan umum dan tin...,6,1,1,0,Kalimantan Tengah,100.0,1 to 2,1 to 2,0 to 5,50%+
4,a23f7c52-9123-46e9-bb5e-be376b1d77f2,2026-07-16T12:38:52+07:00,Psikiater,LEMBAGA PEMASYARAKATAN KELAS III ARJASA,Kab. Sumenep,Bachelor,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,1,0,Jawa Timur,100.0,1 to 2,1 to 2,0 to 5,50%+


## 3.3 Feature Encoding
One-hot encoding `education_level` for downstream stakeholder consumption.

In [42]:
# One hot encode `education_level`
ed_level_dummies = internship_positions["education_level"].str.lower().str.get_dummies(sep=", ")
ed_level_dummies = ed_level_dummies.replace({0: "No", 1: "Yes"}).add_prefix("allows_")
ed_level_dummies = ed_level_dummies.add_suffix("_level")

internship_positions = pd.concat([internship_positions, ed_level_dummies], axis=1)
display(internship_positions.sample(5))

,job_id,published_at,job_title,company,regency_city,education_level,allowed_major,job_description,weekly_working_day,requested_quota,...,applicant_count,province,acceptance_percentage,requested_quota_category,approved_quota_category,applicant_count_category,acceptance_percentage_category,allows_bachelor_level,allows_diploma_level,allows_profession_level
27836,a23dd87e-fa26-496e-98ba-813d6777c191,2026-07-16T10:48:25+07:00,Architectural & Interior Design Intern,Paragon Technology And Innovation,Kota Adm. Jakarta Selatan,"Diploma, Bachelor","Arsitektur, Teknik Arsitektur, Arsitektur Inte...",Job Description,5,1,...,30,DKI Jakarta,3.23,1 to 2,1 to 2,21 to 50,0 - 10%,Yes,Yes,No
18241,a2415765-0da5-4bed-9174-22b24c45fbc5,2026-07-16T12:10:23+07:00,PENGELOLA KEUANGAN DAN ANGGARAN,KANTOR WILAYAH DIREKTORAT JENDERAL IMIGRASI GO...,Kab. Bone Bolango,Bachelor,Akuntansi,"""1. Membantu penyusunan Rencana Kerja dan Angg...",5,2,...,18,Gorontalo,10.53,1 to 2,1 to 2,11 to 20,11 - 25%,Yes,No,No
20343,a242e6fb-9f96-4c7f-927c-0be7c1095acb,2026-07-16T12:36:26+07:00,Asisten Pengelola Keuangan,BPS Kota Sabang,Kota Sabang,"Diploma, Bachelor","Manajemen Keuangan, Ekonomi Pembangunan, Manaj...",Membantu proses pengelolaan administrasi keuan...,5,1,...,11,Aceh,8.33,1 to 2,1 to 2,11 to 20,0 - 10%,Yes,Yes,No
26567,a2419e49-34c1-4e48-a556-2388479c84a8,2026-07-16T12:38:04+07:00,Asisten Analis Sumber Daya Manusia,Kantor Regional XII BKN Pekanbaru,Kota Pekanbaru,Bachelor,"Manajemen, Administrasi Publik, Administrasi P...",Membantu pengumpulan dan verifikasi data kepeg...,5,1,...,21,Riau,4.55,1 to 2,1 to 2,21 to 50,0 - 10%,Yes,No,No
17354,a23f451e-1334-414a-802e-fac9392c80ad,2026-07-16T12:27:40+07:00,Analis Data,Balai Besar Standardisasi dan Pelayanan Jasa I...,Kota Bandung,Bachelor,"Statistik, Statistika","Melaksanakan pengumpulan, pengolahan, analisis...",5,1,...,9,Jawa Barat,10.00,1 to 2,1 to 2,6 to 10,0 - 10%,Yes,No,No


## 3.4 Feature Extraction
Extracting a new binary flag, `allows_all_majors`, by parsing the `job_description` column.

In [43]:
all_majors_condition = internship_positions.job_description.str.contains(
    r"semua\sjurusan|jurusan\sapa.*|all\smajors|any\smajor",
    case=False
)

internship_positions["allows_all_majors"] = np.where(all_majors_condition, "Yes", "No")

display(internship_positions.sample(5))

,job_id,published_at,job_title,company,regency_city,education_level,allowed_major,job_description,weekly_working_day,requested_quota,...,province,acceptance_percentage,requested_quota_category,approved_quota_category,applicant_count_category,acceptance_percentage_category,allows_bachelor_level,allows_diploma_level,allows_profession_level,allows_all_majors
18279,a2434700-7d53-455a-8f67-3eabb5a00ff1,2026-07-16T11:05:44+07:00,Government Relations & Audiency Assistant,Citra Alam Sedayu,Kota Bekasi,"Diploma, Bachelor, Profession","Ilmu Kesejahteraan Sosial, Sosiologi, Administ...",Sebagai Government Relations & Audiency Assist...,5,2,...,Jawa Barat,10.53,1 to 2,1 to 2,11 to 20,11 - 25%,Yes,Yes,Yes,No
18545,a2440826-f449-4cfc-bbef-6c93d7804897,2026-07-16T12:17:34+07:00,BHKS – Asisten Pendampingan dan Advokasi Hukum,Sekretariat Utama,Kota Adm. Jakarta Pusat,"Diploma, Bachelor","Ilmu Komputer, Psikologi, Hukum Pidana/Jinaya...",Mendukung pendampingan hukum BRIN; menyiapkan ...,5,2,...,DKI Jakarta,10.00,1 to 2,1 to 2,11 to 20,0 - 10%,Yes,Yes,No,No
26861,a23f4bac-48f4-45bb-b0c4-9000963428ff,2026-07-16T11:58:23+07:00,PENATA LAYANAN OPERASIONAL,Kantor Wilayah Kementerian Hukum Jawa Barat,Kota Bandung,"Diploma, Bachelor","Manajemen, Manajemen Perkantoran, Ilmu Hukum, ...",Membantu pelaksanaan administrasi pengelolaan ...,5,1,...,Jawa Barat,4.35,1 to 2,1 to 2,21 to 50,0 - 10%,Yes,Yes,No,No
11577,a2376340-de16-4a1a-a8b4-ef005783f065,2026-07-16T10:33:13+07:00,Electrical Engineering,Bambang Djaja,Kota Surabaya,"Diploma, Bachelor","Teknik Elektronika Manufaktur, Teknik Elektron...",Bertanggung jawab mendukung kegiatan engineeri...,5,2,...,Jawa Timur,15.38,1 to 2,1 to 2,11 to 20,11 - 25%,Yes,Yes,No,No
27926,a24166cd-73ba-4d0d-87cd-588306061848,2026-07-16T11:58:13+07:00,Relationship Officer (RO),BPJS Kesehatan Kantor Cabang Medan,Kota Medan,"Diploma, Bachelor","Manajemen, Ilmu Komunikasi, Administrasi Bisni...",Membantu pelaksanaan kegiatan administratif da...,5,1,...,Sumatera Utara,3.03,1 to 2,1 to 2,21 to 50,0 - 10%,Yes,Yes,No,No


# 4. Schema Finalization
Reorganizing the final 14 columns into a logical analytical structure before exporting.

In [44]:
# Get all the columns
internship_positions.columns

Index(['job_id', 'published_at', 'job_title', 'company', 'regency_city',
       'education_level', 'allowed_major', 'job_description',
       'weekly_working_day', 'requested_quota', 'approved_quota',
       'applicant_count', 'province', 'acceptance_percentage',
       'requested_quota_category', 'approved_quota_category',
       'applicant_count_category', 'acceptance_percentage_category',
       'allows_bachelor_level', 'allows_diploma_level',
       'allows_profession_level', 'allows_all_majors'],
      dtype='str')

In [45]:
# Reorganize the position of the columns
final_cols = [
    "job_id",
    "published_at",
    "job_title",
    "company",
    "regency_city",
    "province",
    "allowed_major",
    "allows_all_majors",
    "allows_bachelor_level",
    "allows_diploma_level",
    "allows_profession_level",
    "job_description",
    "weekly_working_day",
    "requested_quota_category",
    "approved_quota_category",
    "applicant_count_category",
    "acceptance_percentage_category",
    "requested_quota",
    "approved_quota",
    "applicant_count",
    "acceptance_percentage",
]

internship_postings = internship_positions[final_cols]

display(internship_postings.sample(10))

,job_id,published_at,job_title,company,regency_city,province,allowed_major,allows_all_majors,allows_bachelor_level,allows_diploma_level,...,job_description,weekly_working_day,requested_quota_category,approved_quota_category,applicant_count_category,acceptance_percentage_category,requested_quota,approved_quota,applicant_count,acceptance_percentage
26491,a2374d99-53d4-4423-b165-8943d2396e98,2026-07-16T11:01:56+07:00,Staf Operasi Keuangan,PT Perusahaan Perseroan (Persero) PT. Pos Indo...,Kota Bandung,Jawa Barat,"Manajemen Keuangan, Akuntansi Manajemen, Manaj...",No,Yes,No,...,Melakukan pemeriksaan dan verifikasi atas kele...,5,1 to 2,1 to 2,21 to 50,0 - 10%,2,2,40,4.88
590,a24116d9-edfd-479e-9606-309dd96f95a0,2026-07-16T12:58:32+07:00,PERAWAT,LEMBAGA PEMASYARAKATAN NARKOTIKA KELAS IIA KAS...,Kab. Katingan,Kalimantan Tengah,Keperawatan,No,Yes,No,...,1. Memberikan perawatan medis dasar bagi pegaw...,6,1 to 2,1 to 2,0 to 5,26 - 50%,1,1,2,33.33
27736,a2411845-05f5-407f-a120-d01c2d51f7fe,2026-07-16T12:11:00+07:00,Asisten Administrator tenaga ahli dibidang IT,OJKI,Kota Palembang,Sumatera Selatan,"Teknik Informatika, Ilmu Komputer, Sistem Info...",No,Yes,Yes,...,Membantu *handling* tools dan teknologi inform...,5,1 to 2,1 to 2,21 to 50,0 - 10%,1,1,29,3.33
10828,a23efbea-ac6f-49d0-b8cf-8403c4c4008e,2026-07-16T10:15:34+07:00,Risk Officer,Pfi Mega Life Insurance,Kota Adm. Jakarta Selatan,DKI Jakarta,"Manajemen, Aktuaria, Statistik, Teknik informa...",No,Yes,No,...,Mendukung tim Risk Management dalam mengelola ...,5,1 to 2,1 to 2,6 to 10,11 - 25%,1,1,6,14.29
16974,a240fb49-a5d8-4326-9abc-6c08b4deaf03,2026-07-16T12:37:35+07:00,Asisten Statistisi,BPS Provinsi Jawa Barat,Kota Bandung,Jawa Barat,"Sains Data, Statistik, Statistika dan Sains Da...",No,Yes,Yes,...,"Membantu pengumpulan, pengolahan, verifikasi, ...",5,3 to 10,3 to 10,21 to 50,11 - 25%,3,3,26,11.11
15785,a2437304-7756-4b92-a831-061987121e4a,2026-07-16T11:11:21+07:00,Content Creator & Social Media,PT Prospect Motor,Kota Samarinda,Kalimantan Timur,"Teknik Informatika, Manajemen, Desain Komunika...",No,Yes,No,...,membuat materi visual untuk kebutuhan pemasara...,6,1 to 2,1 to 2,6 to 10,11 - 25%,1,1,8,11.11
26118,a23edd1b-500a-4b1b-abb2-d9b6f270deec,2026-07-16T10:28:29+07:00,Staf Operasi Pabrik 2,Pupuk Iskandar Muda,Kab. Aceh Utara,Aceh,Teknik Kimia,No,Yes,Yes,...,1. Melakukan inventarisasi potensi kegagalan p...,5,1 to 2,1 to 2,11 to 20,0 - 10%,1,1,19,5.00
22823,a2437036-941b-4c78-b5b3-86540b1ef922,2026-07-16T10:36:46+07:00,R&D,Ligno Specialty Chemicals,Kab. Tangerang,Banten,"Pendidikan Kimia, Kimia Terapan, Teknik Kimia ...",No,Yes,Yes,...,Program pemagangan R&D memberikan pengalaman d...,5,1 to 2,1 to 2,11 to 20,0 - 10%,1,1,13,7.14
15157,a2419073-b046-43ab-82ec-668603c797a8,2026-07-16T13:12:35+07:00,Frontliner (FL),BPJS Kesehatan Kantor Cabang Barabai,Kab. Hulu Sungai Tengah,Kalimantan Selatan,"Administrasi BIsnis, Manajemen, Hubungan Inter...",No,Yes,Yes,...,Membantu pelaksanaan kegiatan administratif da...,5,1 to 2,1 to 2,6 to 10,11 - 25%,1,1,8,11.11
10235,a2251496-d1d3-4ea8-878f-eaa07a10c055,2026-07-16T11:15:43+07:00,Marketing Communication B2B Team 3,Dyandra Promosindo,Kota Adm. Jakarta Pusat,DKI Jakarta,"Manajemen, Pemasaran Digital, Manajemen Pemasa...",No,Yes,Yes,...,Membantu PIC Marcomm dalam berkoordinasi denga...,6,1 to 2,1 to 2,6 to 10,11 - 25%,1,1,6,14.29


In [46]:
# Load the final, clean data to a local directory
internship_postings.to_parquet(
    INTERIM_DATA_DIR / "internship_postings.parquet", index=False
) 